# Demo: Full Prioritization Pipeline

This notebook runs a compact end-to-end demo across the facility cleaning, treatment extraction, survey cleaning, demand scoring, and priority scoring steps.

In [ ]:
from pathlib import Path
import sys

def add_app_python_src_to_path():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "dais-health-access" / "python" / "src"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise FileNotFoundError("Could not find dais-health-access/python/src from the current working directory")

APP_SRC = add_app_python_src_to_path()

In [ ]:
import pandas as pd

from facility_prioritization.config import load_config
from facility_prioritization.facility_processing import clean_facility_data, extract_top_treatments, create_supply_table
from facility_prioritization.survey_processing import clean_survey_data
from facility_prioritization.demand_modeling import calculate_treatment_scores, create_demand_table
from facility_prioritization.priority_scoring import create_priority_table

config = load_config(APP_SRC.parent.parent / "config" / "config.yaml")

raw_facilities = pd.DataFrame(
    [
        {
            "name": "Facility A",
            "latitude": 28.61,
            "longitude": 77.20,
            "specialties": "['cardiology', 'radiology']",
            "procedures": ["angioplasty"],
            "equipment": ["xray", "ultrasound"],
            "description": "cardiac diagnostics",
        },
        {
            "name": "Facility B",
            "latitude": 19.07,
            "longitude": 72.88,
            "specialties": ["oncology", "radiology"],
            "procedures": "chemotherapy",
            "equipment": ["MRI"],
            "description": "cancer treatment center",
        },
        {
            "name": "Facility C",
            "latitude": 13.08,
            "longitude": 80.27,
            "specialties": ["pediatrics"],
            "procedures": ["vaccination"],
            "equipment": ["xray"],
            "description": "child wellness and screening",
        },
    ]
)

raw_survey = pd.DataFrame(
    [
        {"district_name": "Alpha", "state_ut": "State 1", "respiratory_need": "10.5", "cancer_screening_gap": "2.0", "childhood_care_gap": "8.0"},
        {"district_name": "Beta", "state_ut": "State 1", "respiratory_need": "7.1", "cancer_screening_gap": "9.0", "childhood_care_gap": "3.0"},
        {"district_name": "Gamma", "state_ut": "State 2", "respiratory_need": "12.2", "cancer_screening_gap": "4.5", "childhood_care_gap": "6.0"},
    ]
)

geo_reference_df = pd.DataFrame(
    [
        {"district": "Alpha", "latitude": 28.70, "longitude": 77.10, "statename": "State 1"},
        {"district": "Beta", "latitude": 19.20, "longitude": 72.90, "statename": "State 1"},
        {"district": "Gamma", "latitude": 13.00, "longitude": 80.20, "statename": "State 2"},
    ]
)

cleaned_facilities, facility_warnings = clean_facility_data(raw_facilities, config=config)
top_treatments, treatment_warnings = extract_top_treatments(cleaned_facilities, config=config)
supply_table, supply_warnings = create_supply_table(cleaned_facilities, top_treatments, config=config)

cleaned_survey, survey_warnings = clean_survey_data(raw_survey, config=config)

symptom_mapping = pd.DataFrame(
    {
        "respiratory_need": [1, 0, 0],
        "cancer_screening_gap": [0, 1, 0],
        "childhood_care_gap": [0, 0, 1],
        "reasoning": ["Respiratory burden", "Cancer burden", "Child health burden"],
    },
    index=["cardiology", "oncology", "pediatrics"],
)

scores_dict, score_warnings = calculate_treatment_scores(cleaned_survey, symptom_mapping, config=config)
demand_table, demand_warnings = create_demand_table(scores_dict, config=config)
priority_table, priority_warnings = create_priority_table(demand_table, supply_table, geo_reference_df, config=config)

all_warnings = facility_warnings + treatment_warnings + supply_warnings + survey_warnings + score_warnings + demand_warnings + priority_warnings

print("Warnings:")
for warning in all_warnings:
    print(f"- {warning}")

priority_table.head(10)